# TRIBE v2 — Video branch (human vs AI) · NeuroTutorSim

Adapted from Matteo's `00_tribe_demo_v2`. Runs the **video** path (V-JEPA2 video + Wav2Vec audio + Llama transcript) on the **18 matched pairs = 36 clips**, saves each clip's predicted brain response, then compares **human vs AI**.

**Requires Colab Pro** — GPU (A100 or L4) + **High-RAM**. The video path decodes real frames and OOMs on free Colab (that's why Matteo's demo did text only).

## Setup
1. **Runtime → Change runtime type → GPU (A100 or L4) + High-RAM**, then run the install cell.
2. When it finishes: **Runtime → Restart session**, then run the cells **below** the install cell.
3. You need **HuggingFace access to Llama-3.2** (gated) + a token — the login cell will ask for it.
4. Put the 36 `*__clip.mp4` files in Drive at `MyDrive/NeuroTutorSim/video_clips/`.

In [ ]:
# ── Install the exact locked `tribe` stack (Matteo's proven setup) ──
# When this finishes:  Runtime → Restart session,  then run the cells BELOW this one.
import os, subprocess, sys

REPO = "https://github.com/MatteoGuardamagna4/neurotutorsim.git"   # env source (uv.lock + tribe extra); repoint to the neuraimiba fork later
REPO_DIR = "/content/neurotutorsim"
PYTORCH_INDEX = "https://download.pytorch.org/whl/cu124"

def sh(cmd, **kw):
    print("$", " ".join(cmd)); subprocess.run(cmd, check=True, **kw)

sh([sys.executable, "-m", "pip", "install", "-q", "uv"])
if not os.path.isdir(REPO_DIR):
    sh(["git", "clone", "--depth", "1", REPO, REPO_DIR])
sh(["uv", "export", "--frozen", "--extra", "tribe", "--no-hashes", "--no-emit-project",
    "-o", "/tmp/tribe-locked.txt"], cwd=REPO_DIR)
sh(["uv", "pip", "install", "--system", "--index-strategy", "unsafe-first-match",
    "--extra-index-url", PYTORCH_INDEX, "-r", "/tmp/tribe-locked.txt"])
print("\n  Locked tribe stack installed. Now: Runtime → Restart session, then run the cells BELOW.")

$ /usr/bin/python3 -m pip install -q uv
$ git clone --depth 1 https://github.com/MatteoGuardamagna4/neurotutorsim.git /content/neurotutorsim
$ uv export --frozen --extra tribe --no-hashes --no-emit-project -o /tmp/tribe-locked.txt
$ uv pip install --system --index-strategy unsafe-first-match --extra-index-url https://download.pytorch.org/whl/cu124 -r /tmp/tribe-locked.txt

  Locked tribe stack installed. Now: Runtime → Restart session, then run the cells BELOW.


## HuggingFace login (for the gated Llama-3.2 encoder)
Paste a token from an account that has **been granted** access to `meta-llama/Llama-3.2-3B`.

In [ ]:
from huggingface_hub import login
login()   # paste your HF token when prompted

## Load the model

In [ ]:
from tribev2.demo_utils import TribeModel, download_file
from tribev2.plotting import PlotBrain
from pathlib import Path

CACHE_FOLDER = Path("./cache")
model = TribeModel.from_pretrained("facebook/tribev2", cache_folder=CACHE_FOLDER)
plotter = PlotBrain(mesh="fsaverage5")

/usr/local/lib/python3.13/dist-packages/neuralset/extractors/base.py:707: UserWarning: LabelEncoder: event_types has not been set, are you sure you want to apply this extractor to all events?
  warnings.warn(
2026-09-12 08:22:32 - WARNING - neuralset.extractors.base:798 - Missing events will be encoded using the default all-zero value (for example, 0 or a zero vector/tensor), which may be indistinguishable from a valid class if that class is also mapped to zeros. Set treat_missing_as_separate_class=True to avoid this.
/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access pu

config.yaml: 0.00B [00:00, ?B/s]

best.ckpt:   0%|          | 0.00/709M [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/neuralset/extractors/base.py:707: UserWarning: LabelEncoder: event_types has not been set, are you sure you want to apply this extractor to all events?
  warnings.warn(
2026-09-12 08:22:44 - WARNING - neuralset.extractors.base:798 - Missing events will be encoded using the default all-zero value (for example, 0 or a zero vector/tensor), which may be indistinguishable from a valid class if that class is also mapped to zeros. Set treat_missing_as_separate_class=True to avoid this.
INFO - Loading model from /root/.cache/huggingface/hub/models--facebook--tribev2/snapshots/f894e783020944dcd96e5568550afe2aa9743f9f/best.ckpt
INFO:tribev2.demo_utils:Loading model from /root/.cache/huggingface/hub/models--facebook--tribev2/snapshots/f894e783020944dcd96e5568550afe2aa9743f9f/best.ckpt
/usr/local/lib/python3.13/dist-packages/x_transformers/x_transformers.py:439: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('

## Mount Drive + save helpers
Predictions are written to `MyDrive/NeuroTutorSim/tribe_video/` as parquet (rows = TRs, cols = vertices), idempotent (skips clips already done, so a killed run resumes without re-spending compute units).

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd

try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive")
except ModuleNotFoundError:
    DRIVE_ROOT = Path("./drive_local")

OUT_DIR = DRIVE_ROOT / "NeuroTutorSim" / "tribe_video"
OUT_DIR.mkdir(parents=True, exist_ok=True)
CLIPS_DIR = DRIVE_ROOT / "NeuroTutorSim" / "video_clips"   # <-- put the 36 __clip.mp4 files here
print("outputs ->", OUT_DIR)
print("clips   ->", CLIPS_DIR)

def save_preds_parquet(preds, name, overwrite=False):
    path = OUT_DIR / f"{name}_preds.parquet"
    if path.exists() and not overwrite:
        print(f"[skip] {path.name} exists"); return path
    arr = np.asarray(preds)
    df_out = pd.DataFrame(arr, columns=[f"vertex_{v:05d}" for v in range(arr.shape[1])])
    df_out.index.name = "timestep"; df_out.to_parquet(path)
    print(f"[saved] {path.name}  shape={arr.shape}"); return path

Mounted at /content/drive
outputs -> /content/drive/MyDrive/NeuroTutorSim/tribe_video
clips   -> /content/drive/MyDrive/NeuroTutorSim/video_clips


## Fix 1: whisperx word-timing transcription (int8)
The video path extracts audio and runs whisperx for word timings; force `int8` (else it crashes).

In [ ]:
import subprocess as _sp
_orig_run = _sp.run
def _patched_run(cmd, *args, **kwargs):
    if isinstance(cmd, (list, tuple)) and "whisperx" in cmd:
        cmd = list(cmd)
        if "--compute_type" in cmd: cmd[cmd.index("--compute_type")+1] = "int8"
        else: cmd += ["--compute_type", "int8"]
    return _orig_run(cmd, *args, **kwargs)
_sp.run = _patched_run
print("whisperx -> compute_type=int8")

whisperx -> compute_type=int8


## Fix 2: keep the Llama-3.2 text encoder on GPU in fp16
Matteo's OOM fix — stops accelerate from offloading the 3B encoder to system RAM.

In [ ]:
import torch
from transformers import AutoModel
if not getattr(AutoModel, "_oomfp16_patch", False):
    _orig = AutoModel.from_pretrained.__func__
    def _load_on_gpu_fp16(cls, *args, **kwargs):
        if kwargs.get("device_map") == "auto":
            torch.cuda.empty_cache()
            kwargs["device_map"] = {"": 0}; kwargs["torch_dtype"] = torch.float16
            kwargs["low_cpu_mem_usage"] = True
        return _orig(cls, *args, **kwargs)
    AutoModel.from_pretrained = classmethod(_load_on_gpu_fp16); AutoModel._oomfp16_patch = True
    print("Llama encoder -> GPU fp16")
else:
    print("patch already applied")

Llama encoder -> GPU fp16


## Fix 3: NLTK punkt_tab for whisperx
whisperx alignment needs the NLTK `punkt_tab` tokenizer; Colab's proxy blocks nltk's auto-download (SSRF guard), so we fetch it once into a shared path the whisperx subprocess searches.

In [ ]:
# Fix 3: install NLTK punkt_tab for whisperx (robust - nltk with opt-in, else direct download)
import os, nltk
os.environ["NLTK_ALLOW_PROXIED_URLOPEN"] = "1"
try:
    nltk.pathsec.ALLOW_PROXIED_FETCH = True
except Exception:
    pass
DST = "/usr/local/share/nltk_data"
ok = False
try:
    nltk.download("punkt_tab", download_dir=DST)
    nltk.data.find("tokenizers/punkt_tab/english/"); ok = True
except Exception as e:
    print("nltk.download failed, falling back:", e)
if not ok:
    import urllib.request, zipfile, io
    os.makedirs(f"{DST}/tokenizers", exist_ok=True)
    url = "https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/packages/tokenizers/punkt_tab.zip"
    zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(url).read())).extractall(f"{DST}/tokenizers")
    print("punkt_tab installed via direct download")
print("punkt_tab ready ->", DST)

punkt_tab ready -> /usr/local/share/nltk_data


[nltk_data] Downloading package punkt_tab to
[nltk_data]     /usr/local/share/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


## Check the clips are present

In [ ]:
# auto-detect pairs from the clips present (MIT gen_NN, Yale gen_yNN, future) - no hardcoded list
pairs = sorted({f.name[:-len('__human__clip.mp4')] for f in CLIPS_DIR.glob('*__human__clip.mp4')})
missing = [f'{p}__ai' for p in pairs if not (CLIPS_DIR / f'{p}__ai__clip.mp4').exists()]
print(f'{len(pairs)} pairs detected; {len(pairs)-len(missing)} with both human+ai clips')
if missing: print('missing AI clip for:', missing)
if not pairs: print('NO clips found in', CLIPS_DIR, '- check the folder/path')

56 pairs detected; 56 with both human+ai clips


## Run TRIBE on all 36 clips
Each clip → predicted brain response `(n_TRs, ~20k vertices)`. Idempotent: safe to re-run / resume. If a clip OOMs, restart the runtime and re-run this cell (finished clips are skipped).

In [ ]:
import time
for p in pairs:
    for side in ("human", "ai"):
        name = f"{p}__{side}"
        if (OUT_DIR / f"{name}_preds.parquet").exists():
            print(f"[skip] {name}"); continue
        clip = CLIPS_DIR / f"{name}__clip.mp4"
        t0 = time.time()
        df = model.get_events_dataframe(video_path=str(clip))
        preds, segments = model.predict(events=df)
        save_preds_parquet(preds, name=name)
        print(f"[done] {name}  {preds.shape}  {time.time()-t0:.0f}s")
        del df, preds, segments; torch.cuda.empty_cache()

[skip] gen_02__human
[skip] gen_02__ai
[skip] gen_03__human
[skip] gen_03__ai
[skip] gen_04__human
[skip] gen_04__ai
[skip] gen_05__human
[skip] gen_05__ai
[skip] gen_06__human
[skip] gen_06__ai
[skip] gen_07__human
[skip] gen_07__ai
[skip] gen_08__human
[skip] gen_08__ai
[skip] gen_09__human
[skip] gen_09__ai
[skip] gen_10__human
[skip] gen_10__ai
[skip] gen_102__human
[skip] gen_102__ai
[skip] gen_104__human
[skip] gen_104__ai
[skip] gen_106__human
[skip] gen_106__ai
[skip] gen_109__human
[skip] gen_109__ai
[skip] gen_11__human
[skip] gen_11__ai
[skip] gen_110__human
[skip] gen_110__ai


Extract audio from video events: 100%|██████████| 1/1 [00:00<00:00, 1121.17it/s]
/usr/local/lib/python3.13/dist-packages/neuralset/events/transforms/audio.py:56: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  events = pd.concat([events, pd.DataFrame(events_to_add)], ignore_index=True)
/usr/local/lib/python3.13/dist-packages/neuralset/events/utils.py:134: UserWarning: The events dataframe contains an `Index` column. This is dangerous, please add drop=True in calls to df.reset_index(). Dropping it automatically.
  warnings.warn(msg)
Extracting words from audio: 100%|██████████| 1/1 [00:00<00:00,  1.33it/s]
/usr/local/lib/python3.13/dist-packages/neuralset/events/utils.py:134: UserWarning: The events dataframe contains an `Index` column. This i

✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


Add context to words: 100%|██████████| 1293/1293 [00:00<00:00, 60916.75it/s]
[08:26:04 INFO] Preparing extractor: text
INFO:tribev2.main:Preparing extractor: text
Computing word embeddings:   0%|          | 0/324 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...


Extract audio from video events: 100%|██████████| 1/1 [00:03<00:00,  3.26s/it]
/usr/local/lib/python3.13/dist-packages/neuralset/events/transforms/audio.py:56: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  events = pd.concat([events, pd.DataFrame(events_to_add)], ignore_index=True)
/usr/local/lib/python3.13/dist-packages/neuralset/events/utils.py:134: UserWarning: The events dataframe contains an `Index` column. This is dangerous, please add drop=True in calls to df.reset_index(). Dropping it automatically.
  warnings.warn(msg)


MoviePy - Done.


Extracting words from audio: 100%|██████████| 1/1 [00:25<00:00, 25.06s/it]
/usr/local/lib/python3.13/dist-packages/neuralset/events/utils.py:134: UserWarning: The events dataframe contains an `Index` column. This is dangerous, please add drop=True in calls to df.reset_index(). Dropping it automatically.
  warnings.warn(msg)
Add context to words: 100%|██████████| 1134/1134 [00:00<00:00, 47815.39it/s]
[06:13:46 INFO] Preparing extractor: text
INFO:tribev2.main:Preparing extractor: text
Computing word embeddings:   0%|          | 0/284 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Computing word embeddings:  32%|███▏      | 92/284 [07:17<14:05,  4.40s/it]

## Aggregate to Schaefer-400 parcels (matches Matteo / brief schema)
Replicates Matteo's `02_tribe_inference` parcellation: **Schaefer-2018 400-parcel, 7-network** annotation on **fsaverage5**, per-vertex labels `[lh, rh]` (rh offset +200), **unweighted mean over vertices per parcel** → the brief's `stimulus_id, time_index, parcel_id, network, mean_bold, sd_bold`. Self-contained (downloads the annot files directly).

**Verify when the first preds land:** TRIBE's vertex count should be **20484** (fsaverage5, order lh then rh). The aggregation guards on a mismatch.


In [ ]:
import os, urllib.request, numpy as np
try:
    import nibabel as nib
except ImportError:
    os.system('pip -q install nibabel'); import nibabel as nib

ATLAS_DIR = '/content/atlas'; os.makedirs(ATLAS_DIR, exist_ok=True)
BASE = ('https://raw.githubusercontent.com/ThomasYeoLab/CBIG/master/stable_projects/'
        'brain_parcellation/Schaefer2018_LocalGlobal/Parcellations/FreeSurfer5.3/fsaverage5/label')
ann = {}
for hemi in ('lh','rh'):
    fn = f'{hemi}.Schaefer2018_400Parcels_7Networks_order.annot'; dst = f'{ATLAS_DIR}/{fn}'
    if not os.path.exists(dst): urllib.request.urlretrieve(f'{BASE}/{fn}', dst)
    ann[hemi] = nib.freesurfer.read_annot(dst)          # (labels, ctab, names)
lh_lab, _, lh_names = ann['lh']; rh_lab, _, rh_names = ann['rh']
# per-vertex parcel_id: 0 = medial wall/background; lh 1..200, rh 201..400
labels = np.concatenate([lh_lab, np.where(rh_lab > 0, rh_lab + 200, 0)]).astype(int)
names  = [n.decode() if isinstance(n, bytes) else n for n in list(lh_names[1:]) + list(rh_names[1:])]
networks = np.array([nm.split('_')[2] if len(nm.split('_')) > 2 else 'NA' for nm in names])  # 7Networks_LH_Vis_1 -> Vis
N_PARCELS = 400
print('atlas:', labels.shape[0], 'vertices ->', N_PARCELS, 'parcels; networks:', sorted(set(networks)))


atlas: 20484 vertices -> 400 parcels; networks: [np.str_('Cont'), np.str_('Default'), np.str_('DorsAttn'), np.str_('Limbic'), np.str_('SalVentAttn'), np.str_('SomMot'), np.str_('Vis')]


In [ ]:
import pandas as pd
PARCEL_DIR = OUT_DIR.parent / 'tribe_video_parcels'; PARCEL_DIR.mkdir(parents=True, exist_ok=True)
masks = [labels == (i + 1) for i in range(N_PARCELS)]   # vertex mask per parcel

def to_parcels(P, stim):
    T = P.shape[0]
    mean = np.stack([P[:, m].mean(1) if m.any() else np.zeros(T) for m in masks], 1)  # (T,400)
    sd   = np.stack([P[:, m].std(1)  if m.any() else np.zeros(T) for m in masks], 1)
    df = pd.DataFrame({'stimulus_id': stim,
        'time_index': np.repeat(np.arange(T), N_PARCELS),
        'parcel_id':  np.tile(np.arange(1, N_PARCELS + 1), T),
        'network':    np.tile(networks, T),
        'mean_bold':  mean.reshape(-1), 'sd_bold': sd.reshape(-1)})  # brief schema
    return mean, df

parcel_mean = {}                                        # name -> (T,400) mean_bold
for p in pairs:
    for side in ('human', 'ai'):
        name = f'{p}__{side}'; vpath = OUT_DIR / f'{name}_preds.parquet'
        if not vpath.exists(): continue
        P = pd.read_parquet(vpath).values
        if P.shape[1] != labels.shape[0]:
            print(f'[!] {name}: {P.shape[1]} vertices != atlas {labels.shape[0]} - check order/mask'); continue
        mean, df = to_parcels(P, name); df.to_parquet(PARCEL_DIR / f'{name}_parcels.parquet')
        parcel_mean[name] = mean
print(f'aggregated {len(parcel_mean)} clips -> {PARCEL_DIR}')


aggregated 72 clips -> /content/drive/MyDrive/NeuroTutorSim/tribe_video_parcels


## §7 metrics + human-vs-AI contrast (parcel & network level)
Per clip: **mean response, time-to-peak (s), temporal variability, sustained engagement** per parcel. Then per-pair similarity + **group paired contrast (AI − human)** across pairs, at parcel and Yeo-network level.


In [ ]:
from scipy import stats
TR = float(getattr(getattr(globals().get('model', None), 'data', None), 'TR', 1.49))  # TRIBE TR in s; 1.49 fallback
metrics = ('mean', 'ttp', 'var', 'sustained')
def pmetrics(M):                                        # M: (T,400)
    thr = np.median(M, 0)
    return {'mean': M.mean(0), 'ttp': M.argmax(0).astype(float) * TR,
            'var': M.std(0), 'sustained': (M > thr).mean(0)}

rows, diff, done = [], {k: [] for k in metrics}, []
for p in pairs:
    if f'{p}__human' in parcel_mean and f'{p}__ai' in parcel_mean:
        hm, am = pmetrics(parcel_mean[f'{p}__human']), pmetrics(parcel_mean[f'{p}__ai']); done.append(p)
        rows.append({'pair': p, 'meanmap_corr': round(float(np.corrcoef(hm['mean'], am['mean'])[0,1]), 3),
                     'mean_dAI': round(float((am['mean']-hm['mean']).mean()), 4),
                     'ttp_dAI_s': round(float((am['ttp']-hm['ttp']).mean()), 2)})
        for k in metrics: diff[k].append(am[k] - hm[k])
res = pd.DataFrame(rows); print(res.to_string(index=False)); res.to_csv(OUT_DIR/'parcel_human_vs_ai_summary.csv', index=False)
print(f'\n{len(done)} pairs analysed')

for k in metrics:
    if not diff[k]: continue
    G = np.vstack(diff[k]); t, pv = stats.ttest_1samp(G, 0, axis=0)
    np.save(OUT_DIR/f'group_parcel_{k}_diff.npy', G.mean(0)); np.save(OUT_DIR/f'group_parcel_{k}_t.npy', t)
gd = np.vstack(diff['mean']).mean(0)                    # AI - human, mean response, per parcel
tab = pd.DataFrame({'parcel': names, 'network': networks, 'AI_minus_human': np.round(gd, 4)}).sort_values('AI_minus_human')
tab.to_csv(OUT_DIR/'parcel_ranked_diff.csv', index=False)
print('\nParcels: AI drives LESS (top) / MORE (bottom):'); print(pd.concat([tab.head(6), tab.tail(6)]).to_string(index=False))


   pair  meanmap_corr  mean_dAI  ttp_dAI_s
 gen_02         0.949    0.0012      80.63
 gen_03         0.942   -0.0051      49.67
 gen_04         0.963    0.0016     121.73
 gen_05         0.875   -0.0065      61.54
 gen_06         0.940   -0.0049      25.89
 gen_07         0.900   -0.0144      69.85
 gen_08         0.971   -0.0031      54.20
 gen_09         0.893   -0.0210      57.41
 gen_10         0.796   -0.0242      84.74
 gen_11         0.717   -0.0078      77.48
 gen_12         0.901    0.0123     115.74
 gen_13         0.916   -0.0025      57.30
 gen_14         0.894   -0.0318      50.92
 gen_15         0.967    0.0004      46.73
 gen_16         0.949   -0.0149      13.19
 gen_17         0.907    0.0064      83.52
 gen_18         0.862   -0.0174      83.78
 gen_19         0.948    0.0033      97.64
gen_y02         0.860    0.0053      93.62
gen_y03         0.676   -0.0256      46.80
gen_y04         0.888   -0.0090      74.05
gen_y05         0.781   -0.0327      -6.87
gen_y07    

### Covariate control (brief §1.2): adjust for the AI-vs-human feature imbalance
The AI clips are systematically denser/faster/harder (words/min, word count, type-token ratio — all large SMDs). So a raw human-vs-AI *neural* difference could partly be "the AI talks faster," not the teaching regime. Here we regress the per-pair neural difference on the (centered) feature deltas per parcel; the **intercept = the regime effect at the typical feature gap, controlling for pair-to-pair covariate variation**. Upload `feature_balance_table.csv` to Drive `NeuroTutorSim/`.

**Limitation:** the imbalance is large AND collinear with the regime (the AI is *always* denser), so this removes pair-to-pair covariate *variation* but cannot fully separate regime from density (an unobservable zero-gap extrapolation).


In [ ]:
import numpy as np, pandas as pd
from pathlib import Path as _P
fbt_path = next((b/'feature_balance_table.csv' for b in [DRIVE_ROOT/'NeuroTutorSim', _P('/content')]
                 if (b/'feature_balance_table.csv').exists()), None)
if fbt_path is None:
    print('feature_balance_table.csv not found - upload it to Drive NeuroTutorSim/ to run covariate control')
elif not diff['mean']:
    print('no neural diffs yet - run the metrics cell after predictions exist')
else:
    fbt = pd.read_csv(fbt_path).set_index('pair')
    FEATS = ['words_per_min_delta', 'word_count_delta', 'type_token_ratio_delta']
    use = [p for p in done if p in fbt.index]
    if len(use) < 6:
        print(f'only {len(use)} analysed pairs have feature deltas - too few for covariate control')
    else:
        Y = np.vstack([diff['mean'][done.index(p)] for p in use])
        Xraw = fbt.loc[use, FEATS].values.astype(float)
        Xc0 = Xraw - Xraw.mean(0)
        kept, cols, colsraw = [], [], []
        for j, f in enumerate(FEATS):
            col = Xc0[:, j]
            if np.allclose(col, 0):
                print(f'  [drop] {f}: constant across pairs'); continue
            if cols and np.linalg.matrix_rank(np.column_stack(cols + [col])) <= len(cols):
                print(f'  [drop] {f}: collinear with a kept covariate (fixed-length clips)'); continue
            cols.append(col); colsraw.append(Xraw[:, j]); kept.append(f)
        Xc = np.column_stack(cols)
        X = np.column_stack([np.ones(len(use)), Xc])
        beta, *_ = np.linalg.lstsq(X, Y, rcond=None)
        raw = Y.mean(0)
        resid = Y - X @ beta; dof = len(use) - X.shape[1]
        se0 = np.sqrt(((resid**2).sum(0) / dof) * np.linalg.pinv(X.T @ X)[0, 0])
        t_adj = raw / np.where(se0 == 0, np.nan, se0)
        Xu = np.column_stack([np.ones(len(use))] + colsraw)
        bu, *_ = np.linalg.lstsq(Xu, Y, rcond=None); zero_gap = bu[0]
        np.save(OUT_DIR/'group_parcel_mean_t_adjusted.npy', t_adj)
        np.save(OUT_DIR/'group_parcel_mean_zero_gap.npy', zero_gap)
        print(f'covariate model (n={len(use)} pairs, covariates kept: {kept})')
        print(f'  raw group-mean |AI-human| per parcel : {np.abs(raw).mean():.4f}')
        print(f'  mean |t| after covariate adjustment  : {np.nanmean(np.abs(t_adj)):.2f}  (precision, not a new map)')
        print(f'  zero-gap extrapolation |mean|        : {np.abs(zero_gap).mean():.4f}  '
              f'(corr with raw {np.corrcoef(raw, zero_gap)[0,1]:+.2f}; extrapolated - sensitivity only)')
        nd = np.abs(Y).mean(1)
        print('  DOES THE FEATURE GAP EXPLAIN THE NEURAL DIVERGENCE? (the real covariate finding)')
        for f in FEATS:
            r = np.corrcoef(fbt.loc[use, f].values.astype(float), nd)[0, 1]
            print(f'    corr({f:24}, neural_divergence) = {r:+.3f}')

  [drop] word_count_delta: collinear with a kept covariate (fixed-length clips)
covariate-adjusted regime effect (n=36 pairs, covariates kept: ['words_per_min_delta', 'type_token_ratio_delta']):
  raw mean |AI-human| per parcel : 0.0122
  adjusted mean |intercept|      : 0.0122
  corr(raw map, adjusted map)    : +1.000  (high = covariates dont reshape the pattern)
  does the feature gap explain the neural divergence?
    corr(words_per_min_delta     , neural_divergence) = -0.052
    corr(word_count_delta        , neural_divergence) = -0.052
    corr(type_token_ratio_delta  , neural_divergence) = +0.281


### ⭐ Where is the difference — content vs delivery? (network localization)
If the human-vs-AI difference sits in **visual/production** networks and **not** in **language/semantic (Default/Cont)** networks, that's evidence matching isolated *delivery*, not content. If it's in semantic networks, that flags residual content mismatch.


In [ ]:
net_rows = []
for net in sorted(set(networks)):
    m = networks == net
    per_pair = np.array([d[m].mean() for d in diff['mean']])   # AI-human mean diff in this network, per pair
    t, pv = stats.ttest_1samp(per_pair, 0)
    net_rows.append({'network': net, 'n_parcels': int(m.sum()),
                     'AI_minus_human': round(float(per_pair.mean()), 4),
                     't': round(float(t), 2), 'p': round(float(pv), 3)})
net_df = pd.DataFrame(net_rows).sort_values('AI_minus_human'); net_df.to_csv(OUT_DIR/'network_human_vs_ai.csv', index=False)
print(net_df.to_string(index=False))


    network  n_parcels  AI_minus_human     t     p
        Vis         61         -0.0142 -5.95 0.000
   DorsAttn         46         -0.0101 -3.02 0.005
       Cont         52         -0.0079 -2.40 0.022
     Limbic         26         -0.0039 -3.28 0.002
    Default         91         -0.0024 -0.69 0.496
SalVentAttn         47         -0.0014 -0.51 0.615
     SomMot         77          0.0077  3.31 0.002


### ⭐ Does the neural divergence track the human match-rating?
Correlate each pair's **neural divergence** (magnitude of the AI−human parcel difference) with the coders' **1–5 concept-match** rating. Expect a **negative** correlation: worse-matched pairs → bigger neural difference. Put `human_ratings_coder{A,B}.csv` in the Drive `NeuroTutorSim/` folder (or `/content/`).


In [ ]:
import csv
def _ratings(path):
    d = {}
    for r in csv.DictReader(open(path)):
        d[r['pair_id']] = (int(float(r['concept_match_1to5'])), int(float(r['binary_match_0_1'])))
    return d
cand = [DRIVE_ROOT/'NeuroTutorSim', __import__('pathlib').Path('/content')]
RA = RB = None
for base in cand:
    if (base/'human_ratings_coderA.csv').exists() and (base/'human_ratings_coderB.csv').exists():
        RA = _ratings(base/'human_ratings_coderA.csv'); RB = _ratings(base/'human_ratings_coderB.csv'); break
if RA is None:
    print('ratings not found - put human_ratings_coder{A,B}.csv in Drive NeuroTutorSim/ or /content/')
else:
    rated = [p for p in done if p in RA and p in RB]          # only pairs rated by BOTH coders
    if len(rated) < 3:
        print(f'only {len(rated)} analysed pairs are rated - need more ratings for a correlation')
    else:
        nd = [float(np.abs(diff['mean'][done.index(p)]).mean()) for p in rated]   # neural divergence
        hr = [(RA[p][0] + RB[p][0]) / 2 for p in rated]                            # mean 1-5 match
        r = float(np.corrcoef(nd, hr)[0, 1]); rho = stats.spearmanr(nd, hr).correlation
        print(f'{len(rated)} pairs | corr(neural divergence, human 1-5): Pearson {r:+.3f}, Spearman {rho:+.3f}')
        print('(negative = worse match -> bigger neural difference, the expected direction)')
        pd.DataFrame({'pair': rated, 'neural_divergence': np.round(nd,4), 'human_1to5': hr}).to_csv(OUT_DIR/'neural_vs_rating.csv', index=False)

18 pairs | corr(neural divergence, human 1-5): Pearson -0.080, Spearman -0.110
(negative = worse match -> bigger neural difference, the expected direction)


### RDM: representational geometry (brief §7)
Build an 18×18 dissimilarity matrix of the clips' parcel response patterns for **human** and for **AI**, then compare the two geometries (Spearman on the off-diagonal). High = AI preserves the topic-geometry; low = AI homogenizes/reshapes it.


In [ ]:
def rdm(mats): X = np.vstack(mats); return 1 - np.corrcoef(X)      # 1 - Pearson across parcels
Hm = [parcel_mean[f'{p}__human'].mean(0) for p in done]
Am = [parcel_mean[f'{p}__ai'].mean(0)    for p in done]
RH, RAi = rdm(Hm), rdm(Am)
iu = np.triu_indices(len(done), 1)
rho = stats.spearmanr(RH[iu], RAi[iu]).correlation
np.save(OUT_DIR/'rdm_human.npy', RH); np.save(OUT_DIR/'rdm_ai.npy', RAi)
print(f'RDM human-vs-AI geometry: Spearman {rho:+.3f}  (high = same topic-geometry across conditions)')


RDM human-vs-AI geometry: Spearman +0.042  (high = same topic-geometry across conditions)
